# Exercise 04 — SenML
### 📡 The IoT standard for sensor measurements

So far we invented our own JSON structure for sensor data.
In the real world, **SenML** (Sensor Markup Language) is the IETF standard for exactly this.

SenML is compact and designed for constrained devices (think: tiny microcontrollers).

A SenML document is a JSON object with:
- `"bn"` — **base name**: a URI prefix shared by all records (e.g. device ID)
- `"e"` — **events**: array of measurement records

Each event record has:
| Field | Meaning |
|---|---|
| `n` | name (resource name, appended to bn) |
| `v` | value (number) |
| `u` | unit (e.g. `"Cel"`, `"%RH"`, `"ug/m3"`) |
| `t` | timestamp (Unix seconds since 1970-01-01) |

---

## 1️⃣  Your first SenML document

In [ ]:
import json
import time
from datetime import datetime

# A single sensor reading in SenML format
senml = {
    "bn": "http://smartcity.turin.it/sensor/TRN-001/",
    "e": [
        {"n": "temperature", "u": "Cel",   "t": 1700000000, "v": 22.4},
        {"n": "humidity",    "u": "%RH",   "t": 1700000000, "v": 58.0},
        {"n": "pm25",        "u": "ug/m3", "t": 1700000000, "v": 12.1}
    ]
}

print(json.dumps(senml, indent=2))

## 2️⃣  Understand the base name

The full resource name of a measurement is `bn + n`.
Let's reconstruct the full names.

In [ ]:
bn = senml["bn"]

print("Full resource URIs:")
for event in senml["e"]:
    full_name = bn + event["n"]
    ts = datetime.fromtimestamp(event["t"]).strftime("%Y-%m-%d %H:%M:%S")
    print(f"  {full_name}")
    print(f"    → {event['v']} {event['u']}  at {ts}")

## 3️⃣  Generate a time series

SenML shines when sending **multiple timestamped readings** for the same sensor.
Let's simulate 10 minutes of temperature data.

In [ ]:
import random
random.seed(42)

base_time = 1700000000
base_temp = 22.0

events = []
for i in range(10):
    base_temp += random.uniform(-0.3, 0.5)   # small random walk
    events.append({
        "n": "temperature",
        "u": "Cel",
        "t": base_time + i * 60,              # one reading per minute
        "v": round(base_temp, 1)
    })

timeseries = {
    "bn": "http://smartcity.turin.it/sensor/TRN-001/",
    "e": events
}

print(json.dumps(timeseries, indent=2))

## 4️⃣  Parse and analyse a SenML stream

Now let's do the reverse: receive a SenML payload and extract useful info.

In [ ]:
def parse_senml(doc):
    """Extract readings from a SenML document into a list of flat dicts."""
    bn = doc.get("bn", "")
    records = []
    for event in doc["e"]:
        records.append({
            "resource": bn + event["n"],
            "name":     event["n"],
            "value":    event["v"],
            "unit":     event.get("u", ""),
            "time":     datetime.fromtimestamp(event["t"]),
        })
    return records

records = parse_senml(timeseries)

values = [r["value"] for r in records]
print(f"Readings: {len(records)}")
print(f"Min temp : {min(values):.1f} °C")
print(f"Max temp : {max(values):.1f} °C")
print(f"Avg temp : {sum(values)/len(values):.1f} °C")
print()
print("Timeline:")
for r in records:
    bar = '█' * int(r['value'] - 20)   # simple ASCII bar chart
    print(f"  {r['time'].strftime('%H:%M')}  {r['value']:5.1f} °C  {bar}")

## 5️⃣  Multi-sensor SenML — the full network

Each sensor sends its own SenML document. Let's simulate all 5 sensors,
merge their latest readings, and find the hottest spot in the city.

In [ ]:
network_data = [
    {"bn": "http://smartcity.turin.it/sensor/TRN-001/", "e": [{"n": "temperature", "u": "Cel", "t": 1700000060, "v": 22.4}, {"n": "pm25", "u": "ug/m3", "t": 1700000060, "v": 12.1}]},
    {"bn": "http://smartcity.turin.it/sensor/TRN-002/", "e": [{"n": "temperature", "u": "Cel", "t": 1700000060, "v": 23.1}, {"n": "pm25", "u": "ug/m3", "t": 1700000060, "v": 18.4}]},
    {"bn": "http://smartcity.turin.it/sensor/TRN-003/", "e": [{"n": "temperature", "u": "Cel", "t": 1700000060, "v": 21.0}, {"n": "pm25", "u": "ug/m3", "t": 1700000060, "v": 35.2}]},
    {"bn": "http://smartcity.turin.it/sensor/TRN-004/", "e": [{"n": "temperature", "u": "Cel", "t": 1700000060, "v": 24.5}, {"n": "pm25", "u": "ug/m3", "t": 1700000060, "v": 22.7}]},
    {"bn": "http://smartcity.turin.it/sensor/TRN-005/", "e": [{"n": "temperature", "u": "Cel", "t": 1700000060, "v": 20.8}, {"n": "pm25", "u": "ug/m3", "t": 1700000060, "v":  8.3}]},
]

print(f"{'Sensor':<45} {'Measurement':<15} {'Value':>8}")
print("-" * 70)

all_records = []
for doc in network_data:
    for r in parse_senml(doc):
        all_records.append(r)
        flag = " ⚠️" if r["name"] == "pm25" and r["value"] > 20 else ""
        print(f"{r['resource']:<45} {r['name']:<15} {r['value']:>6.1f} {r['unit']}{flag}")

# Find hottest sensor
temps = [r for r in all_records if r["name"] == "temperature"]
hottest = max(temps, key=lambda r: r["value"])
print(f"\n🌡️  Hottest sensor: {hottest['resource']} → {hottest['value']} °C")

## 6️⃣  Compare: our custom JSON vs SenML

Let's see the size difference — important for constrained IoT devices!

In [ ]:
custom_json = {
    "sensor_id": "TRN-001",
    "location": "Piazza Castello",
    "active": True,
    "readings": {"temperature_c": 22.4, "humidity_pct": 58, "pm25_ugm3": 12.1}
}

senml_compact = {
    "bn": "urn:TRN-001:",
    "e": [
        {"n": "t", "u": "Cel",   "t": 1700000000, "v": 22.4},
        {"n": "h", "u": "%RH",   "t": 1700000000, "v": 58.0},
        {"n": "p", "u": "ug/m3", "t": 1700000000, "v": 12.1}
    ]
}

custom_size = len(json.dumps(custom_json))
senml_size  = len(json.dumps(senml_compact))

print(f"Custom JSON : {custom_size} bytes")
print(f"SenML       : {senml_size} bytes")
print(f"Saving      : {custom_size - senml_size} bytes ({100*(custom_size-senml_size)/custom_size:.0f}% smaller)")
print()
print("On a device sending 1 reading/second, that's",
      (custom_size - senml_size) * 86400 // 1024, "KB/day saved.")

---
## 🏆 Challenges

1. Write a `to_senml(sensor_dict)` function that converts the custom JSON format from Exercise 01 into a valid SenML document.
2. Write the reverse: `from_senml(doc)` → returns a flat dict `{"temperature": 22.4, "humidity": 58, ...}`.
3. Use the time series data from step 3 to detect a **temperature spike**: any reading more than 1°C above the previous one.
4. Combine all 5 sensor SenML documents into one big SenML collection and save it to `network_senml.json`.

In [ ]:
# Your code here
